# NB04b — Data Treatment for the Modelling Table

**Role:** Compact-extension  
**Manual section:** 2.2.2.3 — Data preparation and treatment  
**Prerequisites:** NB04

---

## Purpose of this notebook

This compact-extension notebook demonstrates traceable data treatment. It shows how to diagnose issues, stabilise types, map concepts to Python operations, and validate the result — all using the modelling table created in NB04.

**Dataset:** `dataset_C_attrition_model_ready.csv` (created by NB04)

In [ ]:
import pandas as pd
from pathlib import Path

_candidates = [Path('data/processed'), Path('../data/processed')]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Cannot locate data/processed/. Launch from project root or notebooks/.')

MODEL_PATH = DATA_DIR / 'dataset_C_attrition_model_ready.csv'
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        'Shared project dataset not found. Run NB04 first so that '
        '`dataset_C_attrition_model_ready.csv` is created.'
    )

df = pd.read_csv(MODEL_PATH)
print(f'Loaded: {MODEL_PATH}')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## Section 1 — Diagnose before treating

In finance, data treatment should be **traceable and justified**.  
So the first question is never “what can I transform?”, but “what exactly needs treatment?”.

In [ ]:
diagnosis = pd.Series({
    'rows': len(df),
    'unique_customer_ids': df['customer_id'].nunique(),
    'duplicate_rows': int(df.duplicated().sum()),
    'missing_values_total': int(df.isna().sum().sum()),
    'binary_columns': int(df.drop(columns=['customer_id', 'attrition_flag']).isin([0, 1]).all().sum()),
})
diagnosis

In [ ]:
dtype_table = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
})
dtype_table.head(20)

## Section 2 — Apply light, transparent treatment

This modelling table is already in good shape, so treatment is deliberately light:

- ensure binary indicator columns are integer-like;
- ensure the target is integer;
- keep `customer_id` only for traceability, not for modelling;
- keep the file structure stable so later notebooks can reuse it unchanged.

In [ ]:
treated = df.copy()

binary_like = [
    c for c in treated.columns
    if c != 'customer_id' and set(treated[c].dropna().unique()).issubset({0, 1})
]

for c in binary_like:
    treated[c] = treated[c].astype(int)

treated['attrition_flag'] = treated['attrition_flag'].astype(int)

print(f'Binary-like columns normalised: {len(binary_like)}')
treated[binary_like[:8]].head()

## Section 3 — Concept ↔ Python operation

Typical treatment actions and their Python equivalents:

- missing values → `fillna()`
- row aggregation → `groupby()`
- type correction → `astype()` / `pd.to_datetime()`
- joins across sources → `merge()`
- category expansion → `pd.get_dummies()`
- binning / discretisation → `cut()` / `qcut()`

In this notebook, the modelling table mainly needs **validation and type stabilisation**, not heavy repair.

In [ ]:
final_checks = pd.Series({
    'duplicate_rows': int(treated.duplicated().sum()),
    'missing_values_total': int(treated.isna().sum().sum()),
    'target_unique_values': list(sorted(treated['attrition_flag'].unique())),
})
final_checks

In [ ]:
treated.to_csv(MODEL_PATH, index=False)
print(f'Canonical project dataset re-saved to: {MODEL_PATH}')

### Why this matters

The point is not that treatment always means dramatic correction.  
Sometimes the correct action is to **confirm the modelling table is already in good shape**, standardise a few details, and keep a stable file that the next notebooks can reuse.

> Next step: open **NB05**. It loads this exact file to fit the logistic-regression baseline.

---

**Project chain:** NB04 (build table) → **NB04b (treat data)** → NB05 (baseline) → NB05b (trees) → NB06 (validate & interpret)  
**Current position:** NB04b

*This is a compact-extension notebook supporting manual section 2.2.2.3.*